# v126_decoy_groups: decoy-signature features in stage 2 (final version)

| Field | Value |
|---|---|
| **Version** | `v126_decoy_groups` |
| **Plan groups** | C4 / C5 (features), E5 (decision layer) |
| **Parent** | v110 (blocking, stage 1, cached stage-1 outputs of the mock fold and the test split) |
| **Run** | `python experiments/v126_decoy_groups/run_v126.py > run.log` (log committed here) |
| **Final file** | `submissions/v126/`: the x3 arm, uploaded as submission 09 (public F0.5 0.972), the final submission; `submissions/v126_fp6/` is the stricter x6 arm |
| **Reproduce** | `python -m entity_resolution.final` (`make reproduce`): stage A is v110's configuration, stage B this run's, from `src/` |

The run script is the executable pipeline (about 25 minutes on a 4 GB GPU from v110's
caches). This notebook documents it: section 4 prints the functions that ran (imported from
the script), sections 5-7 read back what the run measured and logged, section 9 re-checks the
submitted files.

## 1. Hypothesis

* **Evidence.** Test candidates are far less certain than the mock's (candidates scored
  0.2-0.8 per S1: mock 0.15, India 0.19, US 0.37, France 0.80) and the public score moved only
  with precision. Reading the uncertain test pairs shows the generator's decoy: the S1 name
  plus or minus a word ("Holdings", "Corp", "Comite") at a **nearby house number** (7541 vs
  7532, 26 vs 23), with its own S2 and S3 records. True records carry the S1 number, a
  truncated or zero-padded form of it (201 for 2014, 00326), or one source's own corrupted
  number shared by all of that source's records (600 for 200, three S2 records).
* **Check on the labelled mock (US, kept pairs).** Differing first numbers are true 0.97 of
  the time when one contains the other, 0.65 for a one-digit change, 0.41 within 20; among
  uncertain pairs a number no other candidate holds is true 0.37, shared with one other 0.81.
* **Change.** Stage 2 refitted with columns that read both signals, decoded for the weight the
  public board gives false merges.

## 2. Setup

In [1]:
import importlib.util
import inspect
import json
import subprocess
import sys

import pandas as pd

from entity_resolution import config as C

EXP_DIR = C.EXPERIMENTS / "v126_decoy_groups"
spec = importlib.util.spec_from_file_location("run_v126", EXP_DIR / "run_v126.py")
run = importlib.util.module_from_spec(spec)
spec.loader.exec_module(run)
metrics = json.loads((EXP_DIR / "metrics.json").read_text())["metrics"]
print(run.__doc__)

v126_decoy_groups: stage 2 refit on v110's caches with decoy-signature features.

Test candidates are far less certain than the mock's (uncertain candidates per S1: mock
0.15, India 0.19, US 0.37, France 0.80) and the public score moves only with precision. The
uncertain test pairs show the generator's decoy: the S1 name plus or minus a word at a
*nearby house number* (7541 vs 7532), with its own S2 and S3 records. True records carry the
S1 number, a truncated or padded form of it (201 for 2014), or one source's own corrupted
number shared by all that source's records (600 for 200). On v110's mock (US), differing
first numbers are true 0.97 of the time when one contains the other, 0.65 for a one-digit
change, 0.41 within 20; among uncertain pairs a number no other candidate holds is true 0.37,
shared with one other 0.81. No stage-1 feature reads either signal.

New stage-2 columns on the kept pairs: house-number relation (``hn_*``), candidate groups of
the same S1 (``grp_*``: others sh

## 3. Data

* **Candidates and stage 1:** v110's cached stage-1 outputs (blocking `db1f33c4`: exact,
  name + number and TF-IDF passes, at most 120 per S1; the stage-1 filter keeps p1 >= 0.01
  among each entity's best 16): 4.4 per S1 on the mock, 5.09 per S1 on test.
* **Mock fold:** train reshaped to the test's pool size and pool records per S1 (India 710k S1
  / 4.13M pool, US 663k / 3.82M), dropped S1 leaving their records as unowned decoys (40 % of
  the pool). Stage 2 trains on its fit + tune entities out of fold; the decoder is tuned on
  tune entities and scored on val entities.

## 4. Method

### 4.1 House-number relation of the first address numbers

In [2]:
print(inspect.getsource(run.num_features))

def num_features(hl: np.ndarray, hr: np.ndarray) -> pd.DataFrame:
    """House-number relation of the first address numbers of both sides (NaN if one lacks one).

    ``hn_substr``: one contains the other (truncation, a unit digit dropped); ``hn_onedigit``:
    same length, one digit differs (a noised digit); ``hn_near``: within 20 and neither of
    those (the decoy's neighbouring number); ``hn_missing``: 1 left, 2 right, 3 both lack one.
    """
    both = (hl != "") & (hr != "")
    eq = both & (hl == hr)
    sub = both & ~eq & np.fromiter(((a in b) or (b in a) for a, b in zip(hl, hr, strict=True)),
                                   dtype=bool, count=len(hl))
    one = both & ~eq & np.fromiter(
        (len(a) == len(b) and sum(x != y for x, y in zip(a, b, strict=True)) == 1
         for a, b in zip(hl, hr, strict=True)), dtype=bool, count=len(hl))
    il = pd.to_numeric(pd.Series(hl), errors="coerce").to_numpy(np.float64)
    ir = pd.to_numeric(pd.Series(hr), errors="coerce").to_n

### 4.2 Candidate groups of the same S1 (who else holds the pair's number, name, address)

In [3]:
print(inspect.getsource(run._others_max))
print(inspect.getsource(run.group_features))

def _others_max(key: pd.DataFrame, p1: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Per row: how many other rows share its ``key`` and the best p1 among them (0 if none)."""
    order = np.lexsort((-p1, *[key[c].to_numpy() for c in reversed(key.columns)]))
    k = key.iloc[order].reset_index(drop=True)
    p = p1[order]
    start = np.r_[True, (k.iloc[1:].to_numpy() != k.iloc[:-1].to_numpy()).any(axis=1)]
    grp = np.cumsum(start) - 1
    size = np.bincount(grp)
    first = np.flatnonzero(start)
    top1 = p[first]
    top2 = np.where(size > 1, p[np.minimum(first + 1, len(p) - 1)], 0.0)
    is_first = start
    best_other = np.where(is_first, top2[grp], top1[grp])
    n_other = size[grp] - 1
    out_n, out_best = np.empty(len(p)), np.empty(len(p))
    out_n[order], out_best[order] = n_other, best_other
    return out_n, out_best

def group_features(pairs: pd.DataFrame, p1: np.ndarray, hl: np.ndarray, hr: np.ndarray,
                   name_r: np.ndarray, addr_r: np.ndarray) ->

### 4.3 Unmatched-token IDF (v125's columns) and the assembly on the kept pairs

In [4]:
print(inspect.getsource(run.v125.new_features))
print(inspect.getsource(run.new_features))

def new_features(pairs: pd.DataFrame, s1n: pd.DataFrame, pooln: pd.DataFrame,
                 idf: dict[str, float]) -> pd.DataFrame:
    """The six new columns for ``pairs`` (float32), from normalised name_core / addr_norm."""
    s1 = s1n.set_index(C.ENTITY_ID)
    pool = pooln.set_index(C.ENTITY_ID)
    ln = s1["name_core"].reindex(pairs[C.S1_ID]).fillna("").to_numpy()
    rn = pool["name_core"].reindex(pairs[C.ENTITY_ID]).fillna("").to_numpy()
    la = s1["addr_norm"].reindex(pairs[C.S1_ID]).fillna("").to_numpy()
    ra = pool["addr_norm"].reindex(pairs[C.ENTITY_ID]).fillna("").to_numpy()
    out = np.zeros((len(pairs), len(NEW)), dtype=np.float32)
    default = float(np.median(list(idf.values()))) if idf else 0.0
    for i in range(len(pairs)):
        tl, tr = set(ln[i].split()), set(rn[i].split())
        only_l, only_r = tl - tr, tr - tl
        wl = [idf.get(t, default) for t in only_l]
        wr = [idf.get(t, default) for t in only_r]
        wall = [idf.get(t, default) for

### 4.4 Stage 2
XGBoost on the GPU, 127 leaves, learning rate 0.1, cross-fitted in two parts on the mock's
fit + tune entities (early stopping on each model's held-out part); every entity is scored by
the model that never saw it (test: the mean of both).

In [5]:
print(run.TCFG)
print(json.dumps(metrics['stage2_fit_info'], indent=1)[:1500])

TwoStageConfig(floor=0.01, max_cands=16, folds=2, seed=6161, n_stop_s1=50000, anchors=True, cohesion=False, rivals=True, train_roles=('fit', 'tune'), model=MatcherParams(backend='xgb', num_leaves=127, learning_rate=0.1, n_estimators=3000, early_stopping=60, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, min_data_in_leaf=200, lambda_l2=1.0, max_bin=255, scale_pos_weight=1.0, seed=42, num_threads=12, device='cuda', min_child_weight=1.0), extra_groups=(), drop_columns=())
{
 "fold0": {
  "rows": 2275182,
  "positive_rate": 0.7696052447672318,
  "best_iteration": 598,
  "tune_logloss": 0.043240265595271075,
  "tune_auc": 0.9978800088258298,
  "fit_seconds": 118.62
 },
 "fold1": {
  "rows": 2272479,
  "positive_rate": 0.7699692714432125,
  "best_iteration": 528,
  "tune_logloss": 0.0447615869196099,
  "tune_auc": 0.9976701203008397,
  "fit_seconds": 103.81
 },
 "rows": 4547661,
 "positive_rate": 0.7697871499216851
}


### 4.5 Decision layer
Each pool record stays only with its highest-probability S1; probabilities are
isotonic-calibrated on the tune entities; each entity keeps the prefix of its ranked
candidates that maximises `w·E[F0.5] − (w−1)·E[F0.5 without false positives]`. (w, expected
misses) are selected on tune entities by the tight score at x3 (w 4, miss 0.4: the uploaded
final arm) and at x6 (w 8, miss 0.4), the stricter weight the public steps v122→v123→v126
suggested.

In [6]:
print(inspect.getsource(run.v124.decode_tight))

def decode_tight(scored: pd.DataFrame, w: float, miss: float,
                 max_matches: int = MAX_MATCHES) -> pd.DataFrame:
    """Per-entity prefix maximising the plug-in expected tight score; pool-side 1-to-1 first.

    ``scored``: source1_entity_id, entity_id, prob (calibrated). Returns the kept pairs.
    """
    s = one_to_one_filter(scored[["source1_entity_id", "entity_id", "prob"]])
    s = s.sort_values(["source1_entity_id", "prob"],
                      ascending=[True, False], kind="stable").reset_index(drop=True)
    s1, _ = pd.factorize(s["source1_entity_id"], use_na_sentinel=False)
    q = np.clip(s["prob"].to_numpy(np.float64), EPS, 1.0 - EPS)
    n = s1.max() + 1 if len(s1) else 0
    total = np.bincount(s1, weights=q, minlength=n) + miss     # E[n_true]
    empty = np.exp(np.bincount(s1, weights=np.log1p(-q), minlength=n) - miss)
    start = np.r_[True, s1[1:] != s1[:-1]] if len(s1) else np.zeros(0, dtype=bool)
    starts = np.flatnonzero(start)
    rank = np.aran

## 5. Evaluation on the mock fold (val entities)

Both arms, and v110's own stage-2 scores through the same decoder (the control).

In [7]:
rows = {f"v126 {k}": v for k, v in metrics["arms"].items()}
rows.update({f"v110 control {k}": v for k, v in metrics["arms_v110_control"].items()})
table = pd.DataFrame(rows).T
display(table.drop(columns=["errors"]).astype(float).round(5))
display(pd.DataFrame({k: v["errors"] for k, v in rows.items()}).T)
imp = pd.Series(metrics["top_importance"])
print(f"new columns' gain share {metrics['new_gain_share']:.4f}")
imp.head(15)

,w,miss,f_beta,pair_precision,pair_recall,tight1.45,tight3,tight6
v126 x3,4.0,0.4,0.98281,0.99873,0.95079,0.98223,0.98024,0.97639
v126 x6,8.0,0.4,0.98126,0.99916,0.94451,0.98088,0.97959,0.97708
v110 control x3,4.0,0.4,0.98045,0.99806,0.94421,0.97960,0.97666,0.97098
v110 control x6,8.0,0.4,0.97827,0.99872,0.93582,0.97773,0.97585,0.97222


,false_merge,singleton_merge,missed,false_singleton
v126 x3,1259,158,56205,1698
v126 x6,839,98,63337,1951
v110 control x3,1928,228,63816,1827
v110 control x6,1275,140,73291,2218


new columns' gain share 0.0248


pool_gap              0.670250
p1                    0.164252
s1_gap                0.065996
s1_p1_sum             0.006861
grp_num_p1max         0.005125
pool_p1_sum           0.005117
hn_absdiff            0.004819
pool_best_other       0.004447
uidf_max_r            0.004256
s1_n_likely           0.004094
pool_rank             0.003102
uidf_sum_r            0.002522
sim_name_addr_word    0.001795
num_first_eq          0.001639
riv_addr_gap          0.001609
dtype: float64

## 6. Error analysis

On the mock (v127's audit, same candidates): of 57.8k missed true pairs, 24.9k never become
candidates (heavily noised names with empty or partial addresses, domains, handles), 10.7k go
to a rival S1 in the one-owner step (mostly empty-address records of same-name businesses),
22.2k stay below the rule (probabilities 0.4-0.95). Remaining false merges: an unrelated name
at the S1's address (a co-located business), the S1's name with an empty address, a differing
unit number, a twin at the same address with a legal word added.

## 7. Log

In [8]:
reg = pd.read_csv(C.EXPERIMENTS_CSV, dtype=str).fillna("")
display(reg[reg["version"].isin(["v110", "v123", "v126", "v127", "v128"])][
    ["version", "change", "mock_f05", "public_f05", "commit"]])
print((EXP_DIR / "run.log").read_text()[-2500:])

,version,change,mock_f05,public_f05,commit
16,v110,"v107 two-stage + M3 feature groups (idf, token...",0.9817,0.9660,c9ec108
21,v123,"v122 matcher, rule tuned for false merges x3 (...",0.9823,0.9680,4408d7e
25,v126,"v110 caches; stage 2 + house-number relation, ...",0.9828,0.9720,d4f0fc8
26,v127,stage 3 on v126: competition + group features ...,0.9829,0.9710,78d6b1c
27,v128,"v127 + cohesion, unit-number and name-differen...",0.9821,0.9720,1670490


.9980630201253646, "pair_recall": 0.9442076902155436, "tight1.45": 0.9796007680966745, "tight3": 0.9766648145833043, "tight6": 0.9709823239122649, "errors": {"false_merge": 1928, "singleton_merge": 228, "missed": 63816, "false_singleton": 1827}}
[18:08:10 + 13.5 min] v110 control arm x6: {"w": 8.0, "miss": 0.4, "f_beta": 0.9782734178091949, "pair_precision": 0.9987165137065451, "pair_recall": 0.9358222275107092, "tight1.45": 0.9777282339627419, "tight3": 0.9758503784916265, "tight6": 0.9722158195152737, "errors": {"false_merge": 1275, "singleton_merge": 140, "missed": 73291, "false_singleton": 2218}}
[18:09:16 + 14.6 min] test France: 1,590,100 pairs scored (66 s)
[18:12:16 + 17.6 min] test India: 3,889,020 pairs scored (180 s)
[18:14:38 + 19.9 min] test US: 3,339,480 pairs scored (142 s)
[18:15:50 + 21.1 min] test arm x3: {'pairs': 5686273, 'by_country': {'France': 819245, 'India': 2652987, 'US': 2214041}, 's1_matched': 1629743, 'dir': '/home/suryaguru/StudioProjects/aws/business_enti

## 8. Conclusion

* **Public:** v126 x3 0.972 (v123 0.968): the decoy columns cut mock false merges by a third
  at flat recall and gained 2.5x their mock gain on the public board.
* **Final:** the x3 arm, uploaded as submission 09 (public 0.972), the team's best upload; the
  x6 arm of the same probabilities was the stricter alternative. A third stage (v127, v128)
  gained on the mock but not on public (0.971, 0.972).
* **Next:** recall: 2.1 % of true pairs never become candidates (address-key and name-prefix
  passes).

## 9. Test inference

The final files (`submissions/v126/`, the x3 arm uploaded as submission 09), re-checked with
both validators.

In [9]:
sub = C.ROOT / "submissions" / "v126"
for cmd in ([sys.executable, "-m", "entity_resolution.submission", "--output-dir", str(sub)],
            [sys.executable, str(C.OFFICIAL_VALIDATOR), "--matching",
             str(sub / C.MATCHING_FILE), "--candidate", str(sub / C.CANDIDATE_FILE),
             "--test-dir", str(C.DATASET / "test")]):
    res = subprocess.run(cmd, capture_output=True, text=True)
    print((res.stdout + res.stderr).strip()[-500:])
m_tsv = pd.read_csv(sub / C.MATCHING_FILE, sep="\t", dtype=str, na_filter=False)
n = m_tsv[C.MATCHED_IDS].str.count(",") + (m_tsv[C.MATCHED_IDS] != "")
print(f"{len(m_tsv):,} S1 rows, {int(n.sum()):,} matched pairs, {(n > 0).mean():.4f} of S1 matched")
print(json.dumps(metrics["test"], indent=1))

PASS


2544
  matching_results.tsv: 1732544 rows (102801 empty, 1629743 non-empty).
  candidate_pairs.tsv: 1732544 rows (24603 empty, 1707941 non-empty).

PASS — no blocking issues found. Safe to submit.


1,732,544 S1 rows, 5,686,273 matched pairs, 0.9407 of S1 matched
{
 "x3": {
  "pairs": 5686273,
  "by_country": {
   "France": 819245,
   "India": 2652987,
   "US": 2214041
  },
  "s1_matched": 1629743,
  "dir": "/home/suryaguru/StudioProjects/aws/business_entity_resolution/submissions/v126"
 },
 "x6": {
  "pairs": 5621515,
  "by_country": {
   "France": 806166,
   "India": 2628213,
   "US": 2187136
  },
  "s1_matched": 1627768,
  "dir": "/home/suryaguru/StudioProjects/aws/business_entity_resolution/submissions/v126_fp6"
 }
}
